In [1]:
# import packages
import pandas as pd
import numpy as np
import ot

from tqdm import tqdm
from scipy.stats import norm
import torch
from torch import nn

In [2]:
# Numbers for OT Example in the introduction of the thesis.
print(str(1/2 * np.log(8/2) + 1/2* np.log(8/(2*7))))
print(str(1/2 * np.log(4/2) + 1/2 * np.log(4/(2*3))))
print(str(1/4 *np.log(8/4) + 3/4 * np.log(3/4 * 8/7)))
print(str(1/2 * np.log(4/2) + 1/2 * np.log(4/(2*3))+ 1/4 *np.log(8/4) + 3/4 * np.log(3/4 * 8/7)))

0.4133392865922339
0.14384103622589042
0.05767378526954256
0.20151482149543298


# Jensen Shannon Distance

In [3]:
def JSD(loglik_f_sample_f,
        loglik_f_sample_g,
        loglik_g_sample_f,
        loglik_g_sample_g):
    """
    Jensen-Shannon Distance (JSD) between two probability densities f and g
    parameters:
        loglik_f_sample_f: log likelihood of f evaluated at samples from f
        loglik_f_sample_g: log likelihood of f evaluated at samples from g
        loglik_g_sample_f: log likelihood of g evaluated at samples from f
        loglik_g_sample_g: log likelihood of g evaluated at samples from g
    """
    # calculate ln((f+g)/2) = ln(exp(loglik_f) + exp(loglik_g)) - ln(2)
    ln_avg_loglik_sample_f = np.logaddexp(loglik_f_sample_f, loglik_g_sample_f) - np.log(2)
    ln_avg_loglik_sample_g = np.logaddexp(loglik_f_sample_g, loglik_g_sample_g) - np.log(2)
    # calculate KL(f | (f+g)/2) and KL(g | (f+g)/2)
    KL_f_avg = np.mean(loglik_f_sample_f - ln_avg_loglik_sample_f)
    KL_g_avg = np.mean(loglik_g_sample_g - ln_avg_loglik_sample_g)
    # Calculate Jensen Shannon Distance (Note the square root- not jensen shannon divergence)
    # since the empirical estimate of the JSD is not always positive, max(0, ...) is used
    # to ensure that the square root is always defined.
    jsd = np.sqrt(max(0.5 * (KL_f_avg + KL_g_avg),0.0))
    return jsd

In [4]:
dim=5
repl=1
tau=0.6
par_fun="2quadpf" # "0constpf", "1linpf", "2quadpf", "3cubpf"
last_sim_run_date = "20250821" # date of last simulation run
config_str = f"Dim{dim}Tau{tau}ParFun{par_fun}"
filename_orig = "DistributionDistanceData/" + last_sim_run_date +"Orig" + config_str + "Repl" + str(repl)+ ".csv"
filename_simp_fitted = "DistributionDistanceData/" +last_sim_run_date +"FittedSimp" + config_str + "Repl" + str(repl)+ ".csv"
filename_constpf = "DistributionDistanceData/" +last_sim_run_date +"Constpf" + config_str + "Repl" + str(repl)+ ".csv"
# load_data
orig_file = pd.read_csv(filename_orig).values.astype(np.float32)
simp_fitted_file = pd.read_csv(filename_simp_fitted).values.astype(np.float32)
constpf_file = pd.read_csv(filename_constpf).values.astype(np.float32)
# extract observations
X_obs = orig_file[:,:dim]
X_simp_fitted = simp_fitted_file[:,:dim]
X_constpf = constpf_file[:,:dim]
# extract log likelihoods
loglik_true_obs = orig_file[:,dim]
loglik_fitted_simp_samples_non_simp_vine = orig_file[:,dim+1]
loglik_constpf_samples_non_simp_vine = orig_file[:,dim+2]
loglik_orig_data_simp_fitted = simp_fitted_file[:,dim]
loglik_simp_samples_fitted_simp_vine = simp_fitted_file[:,dim+1]
loglik_orig_data_constpf = constpf_file[:,dim]
loglik_constpf_samples_constpf_vine = constpf_file[:,dim+1]
# computed JSD (Jensen-Shannon Distance)
jsd_value_simp_fitted = JSD(
    loglik_f_sample_f=loglik_true_obs,
    loglik_f_sample_g = loglik_fitted_simp_samples_non_simp_vine,
    loglik_g_sample_f = loglik_orig_data_simp_fitted,
    loglik_g_sample_g= loglik_simp_samples_fitted_simp_vine
)
# jsd_value_constpf = JSD(
#     loglik_f_sample_f=loglik_true_obs,
#     loglik_f_sample_g = loglik_constpf_samples_non_simp_vine,
#     loglik_g_sample_f = loglik_orig_data_constpf,
#     loglik_g_sample_g= loglik_constpf_samples_constpf_vine
# )
print(jsd_value_simp_fitted)
# print(jsd_value_constpf)

0.3464592975890651


# W2

In [5]:
# define the W2 distance function
def W2(x,y):
    return np.sqrt( 
        ot.emd2(
            # approximate the true distribution with the empirical distribution
            np.ones(x.shape[0])/x.shape[0], # empirical distribution of x
            np.ones(y.shape[0])/y.shape[0], # empirical distribution of y
            # squared euclidean distance matrix between x and y
            ot.dist(x, y, metric='sqeuclidean') 
        ) 
    )

In [6]:
# Load dataset from CSV file
csv_path_obs = 'Data/origSamples20250809.csv'
csv_path_sim = 'Data/simpSamples20250809.csv'
X_obs = pd.read_csv(csv_path_obs).values.astype(np.float32)[:1000] # shape (10000, 10)
X_sim = pd.read_csv(csv_path_sim).values.astype(np.float32)[:1000] # shape (10000, 10)


In [7]:
# Compute W2 distance between observed and simulated data
w2_obs = W2(X_obs, X_sim) 
print("The empirical estimate of the Wasserstein distance is: " + str(w2_obs))
# Compute W2 distance between observed data and a random sample of itself as a proxy for the variance of the W2 distance
w2_var = W2(X_obs, X_obs[np.random.permutation(X_obs.shape[0])])  
print("The baseline is: " + str(w2_var))
w2_obs-w2_var  # This gives a measure of how much the observed data deviates from the expected variance in W2 distance

The empirical estimate of the Wasserstein distance is: 0.1859512058508928
The baseline is: 0.00021956119672484573


np.float64(0.18573164465416797)

# MMD

In [19]:
class RBF(nn.Module):
    # RBF is for Radial Basis Function kernel
    def __init__(self, n_kernels=5, mul_factor=2.0, bandwidth=None):
        super().__init__()
        self.bandwidth_multipliers = mul_factor ** (torch.arange(n_kernels) - n_kernels // 2)
        self.bandwidth = bandwidth

    def get_bandwidth(self, L2_distances):
        if self.bandwidth is None:
            # if no bandwidth is provided, compute it as the average of the squared L2 distances
            n_samples = L2_distances.shape[0]
            return L2_distances.data.sum() / (n_samples ** 2 - n_samples)

        return self.bandwidth

    def forward(self, X):
        # Compute the squared L2 distances between all pairs of points in X
        L2_distances = torch.cdist(X, X) ** 2
        # Compute the RBF (Gaussian) kernel using the squared L2 distances and the bandwidth
        # Note that this is averaged over several bandwidths defined by the multipliers
        return torch.exp(-L2_distances[None, ...] / 
                         (self.get_bandwidth(L2_distances) * 
                          self.bandwidth_multipliers)[:, None, None]).sum(dim=0)


class MMDLoss(nn.Module):

    def __init__(self, kernel=RBF()):
        super().__init__()
        self.kernel = kernel

    def forward(self, X, Y):
        K = self.kernel(torch.vstack([X, Y]))

        X_size = X.shape[0]
        XX = K[:X_size, :X_size].mean()
        XY = K[:X_size, X_size:].mean()
        YY = K[X_size:, X_size:].mean()
        return XX - 2 * XY + YY

In [20]:
kernel = RBF()
mmd_loss = MMDLoss(kernel=kernel)

mmd_loss(torch.tensor(X_obs), torch.tensor(X_sim))  # Compute MMD distance between observed and simulated data

tensor(0.0159)

In [42]:
kernel.bandwidth_multipliers

tensor([0.2500, 0.5000, 1.0000, 2.0000, 4.0000])

In [21]:
mmd_loss(torch.tensor(X_obs[:1000,]), torch.tensor(X_obs[:1000,][np.random.permutation(1000)]))

tensor(2.3842e-07)

In [22]:
# Run JSD, MMD Loss and Wasserstein distance on different datasets:
n_repl = 10 
last_sim_run_date = "20250821"
dims = [3,5]
par_funs = ["0constpf", "1linpf", "2quadpf", "3cubpf"]
tau_max = [0.3,0.6,0.9]
total_iterations = len(dims) * len(par_funs) * len(tau_max) * n_repl
# jsd_mat = np.zeros((n_repl, len(dims)*len(par_funs)*len(tau_max), 5))
# wasserstein_mat = np.zeros((n_repl, len(dims)*len(par_funs)*len(tau_max), 6))
# mmd_mat = np.zeros((n_repl, len(dims)*len(par_funs)*len(tau_max),6))
distances_mat = np.zeros((n_repl, len(dims)*len(par_funs)*len(tau_max), 17))  # 17 for all distances and parameters
pbar = tqdm(total=total_iterations, desc="Computing distances")
for r in range(n_repl):
    i=0
    for dim in dims:
        for par_fun_idx in range(len(par_funs)):
            par_fun = par_funs[par_fun_idx]
            for tau in tau_max:
                pbar.update(1)
                data_folder_name = "DistributionDistanceData/"
                config_str = f"Dim{dim}Tau{tau}ParFun{par_fun}Repl{r+1}"
                filename_orig = data_folder_name + last_sim_run_date +"Orig" + config_str + ".csv"
                filename_simp_fitted = data_folder_name +last_sim_run_date +"FittedSimp" + config_str + ".csv"
                filename_constpf = data_folder_name +last_sim_run_date +"Constpf" + config_str + ".csv"
                # load_data
                orig_file = pd.read_csv(filename_orig).values.astype(np.float32)
                simp_fitted_file = pd.read_csv(filename_simp_fitted).values.astype(np.float32)
                constpf_file = pd.read_csv(filename_constpf).values.astype(np.float32)
                # extract observations
                X_obs = orig_file[:,:dim]
                X_simp_fitted = simp_fitted_file[:,:dim]
                X_constpf = constpf_file[:,:dim]
                # extract log likelihoods
                loglik_true_obs = orig_file[:,dim]
                loglik_fitted_simp_samples_non_simp_vine = orig_file[:,dim+1]
                loglik_constpf_samples_non_simp_vine = orig_file[:,dim+2]
                loglik_orig_data_simp_fitted = simp_fitted_file[:,dim]
                loglik_simp_samples_fitted_simp_vine = simp_fitted_file[:,dim+1]
                loglik_orig_data_constpf = constpf_file[:,dim]
                loglik_constpf_samples_constpf_vine = constpf_file[:,dim+1]
                # computed JSD (Jensen-Shannon Distance)
                jsd_value_fitted = JSD(
                    loglik_f_sample_f = loglik_true_obs,
                    loglik_f_sample_g = loglik_fitted_simp_samples_non_simp_vine,
                    loglik_g_sample_f = loglik_orig_data_simp_fitted,
                    loglik_g_sample_g = loglik_simp_samples_fitted_simp_vine
                )
                jsd_value_constpf = JSD(
                    loglik_f_sample_f = loglik_true_obs,
                    loglik_f_sample_g = loglik_constpf_samples_non_simp_vine,
                    loglik_g_sample_f = loglik_orig_data_constpf,
                    loglik_g_sample_g = loglik_constpf_samples_constpf_vine
                )
                # Compute Wasserstein distance
                w2_dist_constpf = W2(X_obs, X_constpf)
                w2_dist_fitted = W2(X_obs, X_simp_fitted)
                w2_baseline = W2(X_obs, X_obs[np.random.permutation(X_obs.shape[0])])
                # Compute MMD
                mmd_value_constpf = mmd_loss(torch.tensor(X_obs), torch.tensor(X_constpf))
                mmd_value_fitted = mmd_loss(torch.tensor(X_obs), torch.tensor(X_simp_fitted))
                mmd_baseline = mmd_loss(torch.tensor(X_obs), torch.tensor(X_obs[np.random.permutation(X_obs.shape[0])]))
                # try on marginally normalized data, otherwise there is no difference between
                # the different scenarios...
                X_obs_normalized = norm.ppf(X_obs)  # marginally normalize the data
                X_simp_fitted_normalized = norm.ppf(X_simp_fitted)  # marginally normalize the data
                X_constpf_normalized = norm.ppf(X_constpf) # marginally normalize the data
                # Compute Wasserstein distance on marginally normalized data
                w2_dist_constpf_normalized = W2(X_obs_normalized, X_constpf_normalized)
                w2_dist_fitted_normalized = W2(X_obs_normalized, X_simp_fitted_normalized)
                w2_baseline_normalized = W2(
                    X_obs_normalized,
                    X_obs_normalized[np.random.permutation(X_obs_normalized.shape[0])]
                )
                # Compute MMD on marginally normalized data
                mmd_value_constpf_normalized = mmd_loss(
                    torch.tensor(X_obs_normalized), 
                    torch.tensor(X_constpf_normalized)
                )
                mmd_value_fitted_normalized = mmd_loss(
                    torch.tensor(X_obs_normalized), 
                    torch.tensor(X_simp_fitted_normalized)
                )
                mmd_baseline_normalized = mmd_loss(
                    torch.tensor(X_obs_normalized), 
                    torch.tensor(X_obs_normalized[np.random.permutation(X_obs_normalized.shape[0])])
                )
                # Store results in matrices
                distances_mat[r,i,] = [
                    dim, tau, par_fun_idx, 
                    jsd_value_constpf, jsd_value_fitted,
                    w2_dist_constpf, w2_dist_fitted, w2_baseline,
                    w2_dist_constpf_normalized, w2_dist_fitted_normalized, w2_baseline_normalized,
                    mmd_value_constpf, mmd_value_fitted, mmd_baseline,
                    mmd_value_constpf_normalized, mmd_value_fitted_normalized, mmd_baseline_normalized
                ]
                # mmd_mat[r,i,] = [dim, par_fun_idx, tau, mmd_value_constpf, mmd_value_fitted, mmd_baseline]
                # jsd_mat[r,i,] = [dim, par_fun_idx, tau, jsd_value_constpf, jsd_value_fitted]
                i += 1
pbar.close()

Computing distances:   5%|▌         | 12/240 [00:33<09:50,  2.59s/it]C:\Users\Michael\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\ot\utils.py:280: RuntimeWarning: invalid value encountered in add
  c += a2[:, None]
C:\Users\Michael\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\ot\lp\__init__.py:630: UserWarning: Problem infeasible. Check that a and b are in the simplex
  check_result(result_code)
C:\Users\Michael\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\ot\utils.py:281: RuntimeWarning: invalid value encountered in add
  c += b2[None, :]
Computing distances:  68%|██████▊   | 162/240 [07:21<03:28,  2.67s/it]C:\Users\Michael\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\ot\ut

In [ ]:
avg_distances_mat = np.mean(distances_mat, axis=0)
sd_distances_mat = np.std(distances_mat, axis=0)
print("Baseline Wasserstein")
print(avg_distances_mat[:,7].max())
print("Baseline Wasserstein Norm")
print(avg_distances_mat[:,10].max())
print("Baseline MMD")
print(avg_distances_mat[:,13])
print("Baseline MMD Norm")
print(avg_distances_mat[:,16])

Baseline Wasserstein
0.00022032549775083992
Baseline Wasserstein Norm
6.31296349427501e-09
Baseline MMD
[-9.53674316e-08  9.53674316e-08  7.15255737e-08  1.43051147e-07
 -1.43051147e-07  1.66893005e-07 -4.76837158e-08 -1.43051147e-07
  7.15255737e-08  2.38418579e-08 -2.38418579e-08  9.53674316e-08
  2.38418579e-08  9.53674316e-08 -1.66893005e-07  2.38418579e-08
  4.76837158e-08  1.19209290e-07 -1.19209290e-07 -1.90734863e-07
 -1.43051147e-07 -1.43051147e-07  1.66893005e-07 -2.38418579e-08]
Baseline MMD Norm
[ 8.88178420e-17 -3.99680289e-16  2.22044605e-16  4.44089210e-17
  3.99680289e-16 -1.77635684e-16  1.77635684e-16  3.10862447e-16
  1.33226763e-16  8.88178420e-17  2.22044605e-16             nan
 -3.55271368e-16  2.22044605e-16  2.22044605e-16 -1.33226763e-16
 -2.66453526e-16             nan  1.33226763e-16  1.77635684e-16
  4.44089210e-16  2.22044605e-16  4.88498131e-16 -2.66453526e-16]


In [24]:
for i in tqdm(range(n_repl)):
    distances_current_repl = distances_mat[i,:,:]
    # Save the distances for each replication
    temp_df = pd.DataFrame(
        data={
        "dim": distances_current_repl[:,0].astype(int),
        "tau": distances_current_repl[:,1],
        "par_fun": [par_funs[i] for i in distances_current_repl[:, 2].astype(int)],
        "jsd_constpf": distances_current_repl[:,3],
        "jsd_fitted": distances_current_repl[:,4],
        "wasserstein_constpf": distances_current_repl[:,5],
        "wasserstein_fitted": distances_current_repl[:,6],
        "wasserstein_baseline": distances_current_repl[:,7],
        "wasserstein_constpf_norm": distances_current_repl[:,8],
        "wasserstein_fitted": distances_current_repl[:,9],
        "wasserstein_baseline_norm": distances_current_repl[:,10],
        "mmd_constpf": distances_current_repl[:,11],
        "mmd_fitted": distances_current_repl[:,12],
        "mmd_baseline": distances_current_repl[:,13],
        "mmd_constpf_norm": distances_current_repl[:,14],
        "mmd_fitted_norm": distances_current_repl[:,15],
        "mmd_baseline_norm": distances_current_repl[:,16]
    }  
    )
    temp_df.to_csv(
        f"Data/{last_sim_run_date}DistancesRepl{i+1}.csv", 
        index=False
    )


100%|██████████| 10/10 [00:00<00:00, 81.92it/s]


In [ ]:
# Latex Table
constpf_distances_df = pd.DataFrame(
    data={
        "dim": avg_distances_mat[:,0].astype(int),
        "tau": avg_distances_mat[:,1],
        "par_fun": [par_funs[i] for i in avg_distances_mat[:, 2].astype(int)],
        "jsd_constpf": avg_distances_mat[:,3],
        "jsd_constpf_stderr": sd_distances_mat[:,3] / np.sqrt(n_repl),
        #"jsd_fitted": avg_distances_mat[:,4],
        "wasserstein_constpf": avg_distances_mat[:,5],
        "wasserstein_constpf_stderr": sd_distances_mat[:,5] / np.sqrt(n_repl),
        #"wasserstein_fitted": avg_distances_mat[:,6],
        #"wasserstein_baseline": avg_distances_mat[:,7],
        #"wasserstein_baseline_stderr": sd_distances_mat[:,7] / np.sqrt(n_repl),
        #"wasserstein_constpf_norm": avg_distances_mat[:,8],
        #"wasserstein_constpf_norm_stderr": sd_distances_mat[:,8] / np.sqrt(n_repl),
        #"wasserstein_fitted": avg_distances_mat[:,9],
        #"wasserstein_baseline_norm": avg_distances_mat[:,10],
        #"wasserstein_baseline_norm_stderr": sd_distances_mat[:,10] / np.sqrt(n_repl),
        "mmd_constpf": avg_distances_mat[:,11],
        "mmd_constpf_stderr": sd_distances_mat[:,11] / np.sqrt(n_repl)
        #"mmd_fitted": avg_distances_mat[:,12],
        #"mmd_baseline": avg_distances_mat[:,13],
        #"mmd_baseline_stderr": sd_distances_mat[:,13] / np.sqrt(n_repl),
        #"mmd_constpf_norm": avg_distances_mat[:,14],
        #"mmd_constpf_norm_stderr": sd_distances_mat[:,14] / np.sqrt(n_repl),
        # "mmd_fitted_norm": avg_distances_mat[:,15],
        #"mmd_baseline_norm": avg_distances_mat[:,16],
        #"mmd_baseline_norm_stderr": sd_distances_mat[:,16] / np.sqrt(n_repl)
    }  
)
constpf_distances_df = constpf_distances_df.sort_values(by=["dim", "tau", "par_fun"])
latex_table_str = (
    constpf_distances_df
    .style
    .hide(axis="index")
    .format({
        "dim": "{:.0f}",           # 0 decimal places
        "tau": "{:.1f}",           # 1 decimal place
        "jsd_constpf": "{:.4f}",   # 3 decimal places
        "jsd_constpf_stderr": "{:.4f}",  # 3 decimal places
        "wasserstein_constpf": "{:.4f}",
        "wasserstein_constpf_stderr": "{:.4f}",
        "mmd_constpf": "{:.4f}",  
        "mmd_constpf_stderr": "{:.4f}"
    })
    .to_latex(hrules=True, column_format="ccccccc")  # hrules=True gives booktabs-style rules
)
print(latex_table_str)

\begin{tabular}{ccccccc}
\toprule
dim & tau & par_fun & jsd_constpf & jsd_constpf_stderr & wasserstein_constpf & wasserstein_constpf_stderr & mmd_constpf & mmd_constpf_stderr \\
\midrule
3 & 0.3 & 0constpf & 0.0000 & 0.0000 & 0.0846 & 0.0008 & 0.0058 & 0.0006 \\
3 & 0.3 & 1linpf & 0.0441 & 0.0051 & 0.0839 & 0.0010 & 0.0048 & 0.0006 \\
3 & 0.3 & 2quadpf & 0.0366 & 0.0069 & 0.0837 & 0.0007 & 0.0045 & 0.0003 \\
3 & 0.3 & 3cubpf & 0.0468 & 0.0059 & 0.0864 & 0.0011 & 0.0057 & 0.0006 \\
3 & 0.6 & 0constpf & 0.0000 & 0.0000 & 0.0833 & 0.0006 & 0.0054 & 0.0006 \\
3 & 0.6 & 1linpf & 0.1169 & 0.0037 & 0.0912 & 0.0012 & 0.0076 & 0.0011 \\
3 & 0.6 & 2quadpf & 0.1424 & 0.0047 & 0.0914 & 0.0014 & 0.0077 & 0.0010 \\
3 & 0.6 & 3cubpf & 0.1285 & 0.0032 & 0.0915 & 0.0012 & 0.0074 & 0.0005 \\
3 & 0.9 & 0constpf & 0.0000 & 0.0000 & 0.0830 & 0.0017 & 0.0071 & 0.0011 \\
3 & 0.9 & 1linpf & 0.2796 & 0.0028 & 0.1018 & 0.0018 & 0.0098 & 0.0012 \\
3 & 0.9 & 2quadpf & 0.2904 & 0.0023 & 0.0938 & 0.0011 & 0.0074 & 

In [ ]:
constpf_distances_df = pd.DataFrame(
    data={
        "dim": avg_distances_mat[:,0].astype(int),
        "tau": avg_distances_mat[:,1],
        "par_fun": [par_funs[i] for i in avg_distances_mat[:, 2].astype(int)],
        "jsd_constpf": avg_distances_mat[:,3],
        "jsd_constpf_stderr": sd_distances_mat[:,3] / np.sqrt(n_repl),
        #"jsd_fitted": avg_distances_mat[:,4],
        "wasserstein_constpf": avg_distances_mat[:,5],
        "wasserstein_constpf_stderr": sd_distances_mat[:,5] / np.sqrt(n_repl),
        #"wasserstein_fitted": avg_distances_mat[:,6],
        "wasserstein_baseline": avg_distances_mat[:,7],
        "wasserstein_baseline_stderr": sd_distances_mat[:,7] / np.sqrt(n_repl),
        "wasserstein_constpf_norm": avg_distances_mat[:,8],
        "wasserstein_constpf_norm_stderr": sd_distances_mat[:,8] / np.sqrt(n_repl),
        #"wasserstein_fitted": avg_distances_mat[:,9],
        "wasserstein_baseline_norm": avg_distances_mat[:,10],
        "wasserstein_baseline_norm_stderr": sd_distances_mat[:,10] / np.sqrt(n_repl),
        "mmd_constpf": avg_distances_mat[:,11],
        "mmd_constpf_stderr": sd_distances_mat[:,11] / np.sqrt(n_repl),
        #"mmd_fitted": avg_distances_mat[:,12],
        "mmd_baseline": avg_distances_mat[:,13],
        "mmd_baseline_stderr": sd_distances_mat[:,13] / np.sqrt(n_repl),
        "mmd_constpf_norm": avg_distances_mat[:,14],
        "mmd_constpf_norm_stderr": sd_distances_mat[:,14] / np.sqrt(n_repl),
        # "mmd_fitted_norm": avg_distances_mat[:,15],
        "mmd_baseline_norm": avg_distances_mat[:,16],
        "mmd_baseline_norm_stderr": sd_distances_mat[:,16] / np.sqrt(n_repl)
    }  
)

latex_table_str = (
    constpf_distances_df
    .style
    .hide(axis="index")
    .format({
        "dim": "{:.0f}",           # 0 decimal places
        "tau": "{:.1f}",           # 1 decimal place
        "mmd_constpf": "{:.3f}",   # 3 decimal places
        "mmd_baseline": "{:.3f}",
        "wasserstein_constpf": "{:.3f}",
        "wasserstein_baseline": "{:.3f}",
        "jsd_constpf": "{:.3f}"
    })
    .to_latex(hrules=True, column_format="ccccccc")  # hrules=True gives booktabs-style rules
)
print(latex_table_str)

\begin{tabular}{ccccccc}
\toprule
dim & tau & par_fun & jsd_constpf & jsd_constpf_stderr & wasserstein_constpf & wasserstein_constpf_stderr & wasserstein_baseline & wasserstein_baseline_stderr & wasserstein_constpf_norm & wasserstein_constpf_norm_stderr & wasserstein_baseline_norm & wasserstein_baseline_norm_stderr & mmd_constpf & mmd_constpf_stderr & mmd_baseline & mmd_baseline_stderr & mmd_constpf_norm & mmd_constpf_norm_stderr & mmd_baseline_norm & mmd_baseline_norm_stderr \\
\midrule
3 & 0.3 & 0constpf & 0.000 & 0.000000 & 0.085 & 0.000820 & 0.000 & 0.000003 & 0.363027 & 0.002825 & 0.000000 & 0.000000 & 0.006 & 0.000586 & -0.000 & 0.000000 & 0.005830 & 0.000556 & 0.000000 & 0.000000 \\
3 & 0.6 & 0constpf & 0.000 & 0.000000 & 0.083 & 0.000646 & 0.000 & 0.000002 & 0.362870 & 0.002240 & 0.000000 & 0.000000 & 0.005 & 0.000581 & 0.000 & 0.000000 & 0.005436 & 0.000595 & -0.000000 & 0.000000 \\
3 & 0.9 & 0constpf & 0.000 & 0.000000 & 0.083 & 0.001721 & 0.000 & 0.000002 & 0.353864 & 0.0043